# Task A_reward: Reward Modelling

In [ ]:
# Importing the required libraries
import os
import torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForSequenceClassification
from huggingface_hub import login
from dotenv import load_dotenv

# Setting the environment variables
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
hf_username = "sohammandal01"

# Seed Setup
torch.manual_seed(42)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# Downloading the dataset
print("Downloading dataset...")
dataset = load_dataset("nrizwan/safe_ai_assignment_1", data_files={"train": "preference_train.csv", "test": "preference_test.csv"})

# Sampling 24k sized dataset to 4k
print("Sub-sampling 4k data points for the Reward Model...")
sampled_dataset = DatasetDict({
    'train': dataset['train'].shuffle(seed=42).select(range(4000)),
    'test': dataset['test'].shuffle(seed=42).select(range(1000))
})
print("Dataset ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

preference_train.csv:   0%|          | 0.00/81.5M [00:00<?, ?B/s]

preference_test.csv:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Sub-sampling 4k data points for the Reward Model...
Dataset ready!


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    chosen_texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(examples["Question"], examples["More_Prefered"])]
    rejected_texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(examples["Question"], examples["Less_Prefered"])]

    tokenized_chosen = tokenizer(chosen_texts, truncation=True, padding="max_length", max_length=512)
    tokenized_rejected = tokenizer(rejected_texts, truncation=True, padding="max_length", max_length=512)

    return {
        "chosen_input_ids": tokenized_chosen["input_ids"],
        "chosen_attention_mask": tokenized_chosen["attention_mask"],
        "rejected_input_ids": tokenized_rejected["input_ids"],
        "rejected_attention_mask": tokenized_rejected["attention_mask"],
    }

print("Tokenizing dataset...")
tokenized_datasets = sampled_dataset.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.select_columns(["chosen_input_ids", "chosen_attention_mask", "rejected_input_ids", "rejected_attention_mask"])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# Custom Reward Trainer
class RewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        rewards_chosen = model(input_ids=inputs["chosen_input_ids"], attention_mask=inputs["chosen_attention_mask"]).logits.squeeze(-1)
        rewards_rejected = model(input_ids=inputs["rejected_input_ids"], attention_mask=inputs["rejected_attention_mask"]).logits.squeeze(-1)

        # Bradley-Terry loss function
        loss = -torch.nn.functional.logsigmoid(rewards_chosen - rewards_rejected).mean()

        if return_outputs:
            return loss, {"rewards_chosen": rewards_chosen, "rewards_rejected": rewards_rejected}
        return loss

def reward_data_collator(features):
    return {
        "chosen_input_ids": torch.stack([torch.tensor(f["chosen_input_ids"]) for f in features]),
        "chosen_attention_mask": torch.stack([torch.tensor(f["chosen_attention_mask"]) for f in features]),
        "rejected_input_ids": torch.stack([torch.tensor(f["rejected_input_ids"]) for f in features]),
        "rejected_attention_mask": torch.stack([torch.tensor(f["rejected_attention_mask"]) for f in features]),
        "labels": torch.zeros(len(features), dtype=torch.float) # Dummy labels so HF Trainer doesn't complain
    }

In [ ]:
# Training the reward model
print("Starting Reward Modelling...")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=1, device_map={"": 0}
)

training_args = TrainingArguments(
    output_dir="./reward_model_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    remove_unused_columns=False,
    fp16=True
)

trainer = RewardTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=reward_data_collator,
)

trainer.train()
print("Reward Modelling Complete!")

# Pushing the reward model to Hugging Face
repo_id = f"{hf_username}/bert-reward-model-assignment"
trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print(f"Reward Model pushed to: {repo_id}")

Starting Reward Modelling...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.062288,0.009271
2,0.011521,0.008358
3,0.008313,0.007980


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ahr_spr/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md: 0.00B [00:00, ?B/s]

Reward Model pushed to: sohammandal01/bert-reward-model-assignment
